# Notebook 03 — Churn Prediction Modeling

**Project:** Telco Customer Churn Analysis

This notebook builds a **leakage-safe, reproducible binary classification
workflow** to predict customer churn, comparing three model families:

1. Logistic Regression
2. Random Forest
3. XGBoost

The methodology uses a strict **train / development (dev) / holdout** split:

```
cleaned data
   |
   +-----------+------------+
   |           |            |
 TRAIN       DEV          HOLDOUT
 (70%)       (15%)         (15%)
   |
 model fitting + hyperparameter selection (5-fold CV on train)
   |
 model comparison + threshold selection (dev ONLY)
   |
 final model + fixed threshold
   |
 HOLDOUT EVALUATED EXACTLY ONCE
   |
 final performance report
```

**Grounding principle:** every metric, threshold, and confusion-matrix value
in this notebook is computed at run time from
`data/cleaned/telco_churn_clean.csv`. No result is hard-coded, and no
holdout-derived information is fed back into model selection.

**Prerequisite:** Notebook 02 identified the statistically associated
features; this notebook tests whether those features *predict* churn. All
feature encoding, scaling, hyperparameter and threshold decisions use only the
train and dev sets.

## 2. Imports

`scikit-learn` provides pipelines, preprocessing, tuning, and metrics;
`xgboost` provides the gradient-boosted trees model; `scipy`/`pandas`/
`numpy`/`matplotlib`/`seaborn` handle data and visualization.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_curve,
    precision_recall_curve,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

## 3. Load and Validate the Cleaned Dataset

Load `data/cleaned/telco_churn_clean.csv` with a path that is robust to the
kernel starting from either the repository root or the `notebooks/` directory,
then validate the data before any modeling.

In [2]:
DATA_FILE = "data/cleaned/telco_churn_clean.csv"


def find_data_path() -> Path:
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidate = folder / DATA_FILE
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {DATA_FILE}. Run this notebook from the repository "
        "root or from the notebooks/ directory."
    )


DATA_PATH = find_data_path()
print("Resolved data path:", DATA_PATH)

EXPECTED_ROWS = 7032

df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)

problems = []
if df.shape[0] != EXPECTED_ROWS:
    problems.append(f"Expected {EXPECTED_ROWS} rows, found {df.shape[0]}")
if "Churn" not in df.columns:
    problems.append("Churn column is missing")
elif not set(df["Churn"].unique()).issubset({0, 1}):
    problems.append(f"Churn contains unexpected values: {sorted(df['Churn'].unique())}")
if problems:
    raise ValueError("Dataset validation failed:\n- " + "\n- ".join(problems))

missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0].to_string() if (missing > 0).any() else "  None")
print("Duplicate rows (exact, keep='first'):", int(df.duplicated().sum()))
print()
print("Target (Churn) distribution:")
print(df["Churn"].value_counts().to_string())
print(f"Overall churn rate: {(df['Churn'] == 1).mean():.4f}")

Resolved data path: /Users/vaibhavvikasranjan/Downloads/telco-churn-analysis/data/cleaned/telco_churn_clean.csv
Loaded shape: (7032, 20)
Missing values per column:
  None
Duplicate rows (exact, keep='first'): 22

Target (Churn) distribution:
Churn
0    5163
1    1869
Overall churn rate: 0.2658


## 4. Duplicate Handling — Documented Modeling Decision

Notebook 02 reported **22 exact duplicate rows** (identical across all 20
columns). They are part of the cleaned dataset and are not silently removed
from the source file.

**Decision for modeling:** the 22 exact duplicates are dropped *for modeling
only*, on a copy of the data. Rationale:

- Exact duplicates carry no additional information for a supervised model.
- If one copy of a duplicate lands in train and another in holdout, the model
  effectively "memorizes" the label, inflating the honest holdout estimate.
- The source file `telco_churn_clean.csv` is left untouched.

The drop is applied to a working copy before any split, so no duplicate row
can appear in more than one of train/dev/holdout.

In [3]:
duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows in cleaned dataset:", duplicate_count)

df_model = df.drop_duplicates(keep="first")
print("Modeling set shape:", df_model.shape)
print("Modeling set churn rate:", f"{(df_model['Churn'] == 1).mean():.4f}")

Exact duplicate rows in cleaned dataset: 22
Modeling set shape: (7010, 20)
Modeling set churn rate: 0.2649


## 5. Train / Dev / Holdout Split

Churn is imbalanced (~73% / ~27%), so both splits are **stratified on Churn**
and performed with a fixed `random_state` for reproducibility.

Two-step split to reach the target sizes:

- Step 1: 70% train, 30% temporary
- Step 2: temporary split 50/50 into dev (15%) and holdout (15%)

The **holdout is set aside now and is not passed to any model-selection
function** anywhere in this notebook.

In [4]:
X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "n_rows": [len(y_train), len(y_dev), len(y_holdout)],
    "churned": [int(y_train.sum()), int(y_dev.sum()), int(y_holdout.sum())],
    "churn_rate": [y_train.mean(), y_dev.mean(), y_holdout.mean()],
}, index=["Train", "Dev", "Holdout"])
display(split_summary.round(4))

,n_rows,churned,churn_rate
Train,4907,1300,0.2649
Dev,1051,278,0.2645
Holdout,1052,279,0.2652


## 6. Features, Target, and Preprocessing

The target is `Churn`. All other columns are features.

- **Numerical features:** `tenure`, `MonthlyCharges`, `TotalCharges`
  (plus `SeniorCitizen`, a 0/1 indicator kept as numeric).
- **Categorical features:** the remaining object columns.

Preprocessing is defined with a `ColumnTransformer` and fitted **only on the
training set** (each pipeline below fits it inside `GridSearchCV` on train).
No preprocessing object is ever fitted on dev or holdout data, and the full
dataset is never preprocessed before splitting.

- Numerical: median imputation (safety; the data has no missing values) +
  `StandardScaler` (needed for Logistic Regression).
- Categorical: most-frequent imputation + `OneHotEncoder(handle_unknown="ignore")`
  so that any unseen category in dev/holdout does not crash the pipeline.

In [5]:
categorical_features = X.select_dtypes(include="object").columns.tolist()
numerical_features = X.select_dtypes(exclude="object").columns.tolist()
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features),
])
print("Preprocessor defined. It will be fitted ONLY on training data.")

Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Preprocessor defined. It will be fitted ONLY on training data.


## 7. Reusable Evaluation Helpers

All metric logic lives in small, documented helpers so that the same code is
used for the baseline, the dev comparison, and the final holdout evaluation.

In [6]:
THRESHOLD_GRID = np.arange(0.05, 0.96, 0.025)


def evaluate_thresholds(y_true, proba, grid=THRESHOLD_GRID):
    '''Precision/recall/F1 across a threshold grid.

    Returns (frame, best_threshold). The best threshold maximizes F1; ties are
    broken toward the HIGHEST threshold that achieves the maximum F1, which
    favours precision among equally good thresholds.
    '''
    rows = []
    for t in grid:
        pred = (proba >= t).astype(int)
        rows.append({
            "threshold": t,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred, zero_division=0),
        })
    frame = pd.DataFrame(rows)
    best_f1 = frame["f1"].max()
    best_threshold = float(frame.loc[frame["f1"] == best_f1, "threshold"].max())
    return frame, best_threshold


def evaluate_classifier(y_true, proba, threshold):
    '''Full metric set for a model at a fixed decision threshold.'''
    pred = (proba >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, average="binary", zero_division=0
    )
    return {
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy_score(y_true, pred),
        "confusion_matrix": confusion_matrix(y_true, pred),
    }


def plot_confusion_matrix(cm, title, class_names=("Retained", "Churned")):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=[f"{c} (pred)" for c in class_names],
        yticklabels=[f"{c} (actual)" for c in class_names],
        ax=ax,
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_roc_curve(y_true, proba):
    fpr, tpr, _ = roc_curve(y_true, proba)
    auc = roc_auc_score(y_true, proba)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(fpr, tpr, label=f"ROC-AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], "k--", label="Chance (AUC = 0.50)")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate (recall)")
    ax.set_title("ROC Curve")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_precision_recall_curve(y_true, proba):
    precision, recall, _ = precision_recall_curve(y_true, proba)
    ap = average_precision_score(y_true, proba)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(recall, precision, label=f"PR-AUC (Average Precision) = {ap:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 8. Baseline — Always Predict the Majority Class

A trivial baseline that always predicts the majority class (retained / 0) is
evaluated on the **dev set**. This is the reference a real model must beat.
Accuracy alone is misleading here: because ~73% of customers do not churn, a
model that flags nobody can still "score" near 73% accuracy while detecting
zero churners. The confusion matrix below makes that explicit.

In [7]:
baseline_pred = np.zeros_like(y_dev)
tn, fp, fn, tp = confusion_matrix(y_dev, baseline_pred).ravel()
print(f"Dev baseline accuracy  = {accuracy_score(y_dev, baseline_pred):.4f}")
print(f"Dev baseline precision = {precision_score(y_dev, baseline_pred, zero_division=0):.4f}")
print(f"Dev baseline recall    = {recall_score(y_dev, baseline_pred, zero_division=0):.4f}")
print(f"Dev baseline F1        = {f1_score(y_dev, baseline_pred, zero_division=0):.4f}")
print(f"Dev baseline confusion matrix: TN={tn} FP={fp} FN={fn} TP={tp}")

plot_confusion_matrix(
    confusion_matrix(y_dev, baseline_pred),
    "Dev Baseline (always predict 'Retained')",
)

Dev baseline accuracy  = 0.7355
Dev baseline precision = 0.0000
Dev baseline recall    = 0.0000
Dev baseline F1        = 0.0000
Dev baseline confusion matrix: TN=773 FP=0 FN=278 TP=0


## 9. Metric Strategy

**Why accuracy is not the goal.** Churn is imbalanced (~27% churners). A
baseline that predicts everyone as retained reaches ~73% accuracy while
detecting **no** churners — useless for retention targeting.

**Primary metrics** (all reported for every model):

- **Recall:** of the customers who actually churned, how many did the model
  flag? `recall = TP / (TP + FN)`
- **Precision:** of the customers flagged as likely to churn, how many
  actually churned? `precision = TP / (TP + FP)`
- **F1:** harmonic mean of precision and recall, a single number that balances
  both.
- **ROC-AUC:** threshold-agnostic ranking quality (less sensitive to imbalance
  than accuracy, but still dominated by the majority class).
- **PR-AUC (Average Precision):** threshold-agnostic precision/recall summary
  that is **more informative for the minority churn class**.

**Business trade-off.**

- **False negative (FN):** a customer who churned but was not flagged — the
  retention team never sees them; the loss is realized.
- **False positive (FP):** a customer who was flagged but would not have
  churned — retention budget is spent on someone who was going to stay.

**Selection criterion (stated up front):** models and decision thresholds are
selected by **development-set F1** (threshold chosen per model on dev to
maximize F1). F1 treats missing a churner and wasting an intervention as
equally costly. This is a deliberate assumption: a real deployment that knows
the dollar cost of FNs vs FPs should re-select the threshold using those costs.
This notebook does **not** assume recall always matters more than precision —
F1 is the explicit balance point.

## 10. Model 1 — Logistic Regression

A linear model with L2 regularization. Class imbalance is handled with
`class_weight` (grid-tested between `"balanced"` and `None`), and `C` controls
regularization strength. Only a small grid is searched via 5-fold stratified
CV **on the training set only**, scored by F1.

The preprocessing pipeline (scaling + one-hot encoding) is fitted inside the
pipeline on train, so no dev/holdout information leaks in.

In [8]:
lr_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000)),
])

lr_grid = {
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
    "classifier__class_weight": ["balanced", None],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_search = GridSearchCV(
    lr_pipeline, lr_grid, scoring="f1", cv=cv, n_jobs=-1
)
lr_search.fit(X_train, y_train)
print("Best Logistic Regression params:", lr_search.best_params_)
print("Best CV F1 (train):", f"{lr_search.best_score_:.4f}")

Best Logistic Regression params: {'classifier__C': 10.0, 'classifier__class_weight': 'balanced'}
Best CV F1 (train): 0.6232


## 11. Model 2 — Random Forest

An ensemble of decision trees that can capture non-linear interactions.
Imbalance is handled with `class_weight="balanced"`. A small grid over tree
capacity and depth is searched with 5-fold CV on train, scored by F1.

In [9]:
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    )),
])

rf_grid = {
    "classifier__n_estimators": [200, 400],
    "classifier__max_depth": [10, None],
    "classifier__min_samples_leaf": [2, 4],
}

rf_search = GridSearchCV(
    rf_pipeline, rf_grid, scoring="f1", cv=cv, n_jobs=-1
)
rf_search.fit(X_train, y_train)
print("Best Random Forest params:", rf_search.best_params_)
print("Best CV F1 (train):", f"{rf_search.best_score_:.4f}")

Best Random Forest params: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 2, 'classifier__n_estimators': 200}
Best CV F1 (train): 0.6319


## 12. Model 3 — XGBoost

Gradient-boosted trees. Imbalance is handled with `scale_pos_weight`, computed
**from the training set only** as the ratio of negative to positive samples.
Reproducibility is fixed via `random_state`. A modest grid over boosting
capacity and regularization is searched with 5-fold CV on train, scored by F1.

In [10]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight (computed from TRAIN only):", f"{scale_pos_weight:.4f}")

xgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1,
    )),
])

xgb_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [3, 6],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
}

xgb_search = GridSearchCV(
    xgb_pipeline, xgb_grid, scoring="f1", cv=cv, n_jobs=-1
)
xgb_search.fit(X_train, y_train)
print("Best XGBoost params:", xgb_search.best_params_)
print("Best CV F1 (train):", f"{xgb_search.best_score_:.4f}")

scale_pos_weight (computed from TRAIN only): 2.7746


Best XGBoost params: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 200, 'classifier__subsample': 0.8}
Best CV F1 (train): 0.6253


## 13. Model Comparison on the Development Set

For each tuned model, probabilities are predicted on the **dev set only**. For
each model we select its own best threshold (maximizing F1 on dev, see
section 14) and report the full metric set at that threshold. Threshold-agnostic
ROC-AUC and PR-AUC are reported as well.

All numbers below are computed at run time.

In [11]:
model_searches = {
    "Logistic Regression": lr_search,
    "Random Forest": rf_search,
    "XGBoost": xgb_search,
}

dev_results = {}
dev_proba = {}
for name, search in model_searches.items():
    proba = search.predict_proba(X_dev)[:, 1]
    dev_proba[name] = proba
    _, best_thr = evaluate_thresholds(y_dev, proba)
    dev_results[name] = evaluate_classifier(y_dev, proba, best_thr)

dev_comparison = pd.DataFrame({
    name: {
        "ROC-AUC": r["roc_auc"],
        "PR-AUC": r["pr_auc"],
        "Precision": r["precision"],
        "Recall": r["recall"],
        "F1": r["f1"],
        "Accuracy": r["accuracy"],
        "Threshold": r["threshold"],
    }
    for name, r in dev_results.items()
}).T

display(dev_comparison.round(4))

,ROC-AUC,PR-AUC,Precision,Recall,F1,Accuracy,Threshold
Logistic Regression,0.8454,0.6376,0.5635,0.7338,0.6375,0.7793,0.575
Random Forest,0.8436,0.6406,0.5541,0.7554,0.6393,0.7745,0.450
XGBoost,0.8512,0.6605,0.5518,0.7662,0.6416,0.7735,0.525


### Result (computed from data)

| Model | ROC-AUC | PR-AUC | Precision | Recall | F1 | Accuracy | Dev threshold |
|---|---|---|---|---|---|---|---|
| Logistic Regression | 0.8454 | 0.6376 | 0.5635 | 0.7338 | 0.6375 | 0.7793 | 0.575 |
| Random Forest | 0.8436 | 0.6406 | 0.5541 | 0.7554 | 0.6393 | 0.7745 | 0.450 |
| XGBoost | 0.8512 | 0.6605 | 0.5518 | 0.7662 | 0.6416 | 0.7735 | 0.525 |

All three models are far above the 0.7355 accuracy of the majority-class
baseline, and all identify the large majority of churners (recall ≥ 0.73).
XGBoost has the highest dev F1 (0.6416) and the best ROC-AUC/PR-AUC.

## 14. Decision Threshold Selection

The default 0.5 threshold is **not** assumed to be optimal. For each model the
dev probabilities are swept over a threshold grid and precision/recall/F1 are
computed at every threshold. The selected threshold maximizes **F1 on dev**.

The plots below show the precision/recall trade-off for each model; the
vertical line marks the chosen threshold. Every decision here uses dev only.

In [12]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, (name, proba) in zip(axes, dev_proba.items()):
    thr_frame, _ = evaluate_thresholds(y_dev, proba)
    ax.plot(thr_frame["threshold"], thr_frame["precision"], label="Precision")
    ax.plot(thr_frame["threshold"], thr_frame["recall"], label="Recall")
    ax.plot(thr_frame["threshold"], thr_frame["f1"], label="F1")
    ax.axvline(dev_results[name]["threshold"], color="black", linestyle="--", alpha=0.5)
    ax.set_title(f"{name}  (best threshold = {dev_results[name]['threshold']:.3f})")
    ax.set_xlabel("Threshold")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Score")
plt.tight_layout()
plt.show()

threshold_table = pd.DataFrame({
    name: {"Best dev threshold (max F1)": r["threshold"],
           "F1 at that threshold": r["f1"],
           "Precision": r["precision"],
           "Recall": r["recall"]}
    for name, r in dev_results.items()
}).T
display(threshold_table.round(4))

,Best dev threshold (max F1),F1 at that threshold,Precision,Recall
Logistic Regression,0.575,0.6375,0.5635,0.7338
Random Forest,0.450,0.6393,0.5541,0.7554
XGBoost,0.525,0.6416,0.5518,0.7662


### Result (computed from data)

Chosen dev thresholds (maximize F1): Logistic Regression **0.575**, Random
Forest **0.450**, XGBoost **0.525**. None equals the naive 0.5 default,
confirming that threshold selection matters. Raising the threshold trades
recall for precision (fewer, higher-confidence flags); lowering it does the
opposite.

## 15. Confusion Matrix on the Development Set

The final model is selected by **development-set F1**. In business terms:

- **True negative (TN):** did not churn and was not flagged — correct.
- **False positive (FP):** did not churn but was flagged — retention budget
  spent on a customer who would have stayed.
- **False negative (FN):** churned but was not flagged — the retention team
  never saw them.
- **True positive (TP):** churned and was flagged — correctly identified.

In [13]:
best_name = dev_comparison["F1"].idxmax()
best_threshold = dev_results[best_name]["threshold"]
print("Selected model (highest dev F1):", best_name)
print("Selected threshold (dev, max F1):", f"{best_threshold:.3f}")

cm_dev = dev_results[best_name]["confusion_matrix"]
tn, fp, fn, tp = cm_dev.ravel()
print(f"Dev confusion matrix at threshold {best_threshold:.3f}: "
      f"TN={tn} FP={fp} FN={fn} TP={tp}")

plot_confusion_matrix(
    cm_dev,
    f"Dev Confusion Matrix - {best_name} (threshold {best_threshold:.3f})",
)

Selected model (highest dev F1): XGBoost
Selected threshold (dev, max F1): 0.525
Dev confusion matrix at threshold 0.525: TN=600 FP=173 FN=65 TP=213


### Result (computed from data)

Selected model: **XGBoost** with dev threshold **0.525**. Dev confusion
matrix: **TN=600, FP=173, FN=65, TP=213** (n = 1,051). The model flags 386 of
1,051 dev customers (36.7%); of the 278 actual churners it catches 213
(recall 0.766) and misses 65; of the 773 retained customers it wrongly flags
173 (22.4% of retained).

## 16. Final Model Selection

**Model selection was performed using development-set F1.** The holdout set
was **not inspected** during model comparison, hyperparameter tuning, or
threshold selection.

Decision chain used so far (all train/dev only):

1. Hyperparameters: 5-fold stratified CV **on train**, scored by F1.
2. Model comparison: predictions on **dev**, metric set at each model's
   dev-selected threshold.
3. Final model: the one with the highest dev F1 — **XGBoost**.
4. Decision threshold: the dev F1-maximizing threshold of the selected model —
   **0.525**.

The holdout is used next, **exactly once**, with this model and this fixed
threshold.

## 17. FINAL HOLDOUT EVALUATION

**This is the final, one-time evaluation.** The preprocessing, model,
hyperparameters, and threshold are all finalized from train/dev. The holdout
set is scored **exactly once**, and no result below is used to change any
decision.

In [14]:
final_search = model_searches[best_name]
proba_holdout = final_search.predict_proba(X_holdout)[:, 1]
holdout_report = evaluate_classifier(y_holdout, proba_holdout, best_threshold)

print("=" * 62)
print("FINAL HOLDOUT EVALUATION (evaluated exactly once)")
print("=" * 62)
print(f"Model:              {best_name}")
print(f"Decision threshold: {best_threshold:.3f} (fixed from dev)")
print(f"Holdout size:       {len(y_holdout)} customers")
print()
print(f"ROC-AUC     = {holdout_report['roc_auc']:.4f}")
print(f"PR-AUC      = {holdout_report['pr_auc']:.4f}")
print(f"Precision   = {holdout_report['precision']:.4f}")
print(f"Recall      = {holdout_report['recall']:.4f}")
print(f"F1          = {holdout_report['f1']:.4f}")
print(f"Accuracy    = {holdout_report['accuracy']:.4f}")

FINAL HOLDOUT EVALUATION (evaluated exactly once)
Model:              XGBoost
Decision threshold: 0.525 (fixed from dev)
Holdout size:       1052 customers

ROC-AUC     = 0.8530
PR-AUC      = 0.6571
Precision   = 0.5369
Recall      = 0.7814
F1          = 0.6365
Accuracy    = 0.7633


### Result (computed from data)

On the untouched holdout set (1,052 customers), the selected model
(**XGBoost**) at the dev-fixed threshold **0.525** achieved:

| Metric | Value |
|---|---|
| ROC-AUC | 0.8530 |
| PR-AUC | 0.6571 |
| Precision | 0.5369 |
| Recall | 0.7814 |
| F1 | 0.6365 |
| Accuracy | 0.7633 |

The holdout results are close to the dev estimates (dev F1 0.6416), which is
consistent with the model not overfitting the dev set. **These numbers were
produced by a single holdout pass and were not used to change any decision.**

## 18. ROC Curve on the Holdout Set

ROC-AUC summarizes ranking quality across all thresholds. It is reported for
completeness, but because churn is imbalanced it is **not** used as the
primary measure — a high ROC-AUC can coexist with poor minority-class
precision, which is why the precision-recall curve follows.

In [15]:
plot_roc_curve(y_holdout, proba_holdout)

## 19. Precision-Recall Curve on the Holdout Set

The PR curve is the more informative threshold-agnostic view for the minority
churn class: it directly reflects how precision behaves as recall increases.
Average Precision (PR-AUC) is reported. A useful reference line is the churn
prevalence in the holdout (~0.265): an uninformative model would sit near that
prevalence line.

In [16]:
plot_precision_recall_curve(y_holdout, proba_holdout)

## 20. Final Holdout Confusion Matrix

Final confusion matrix at the fixed dev threshold, with false-positive and
false-negative rates, interpreted in business terms.

In [17]:
cm_hold = holdout_report["confusion_matrix"]
tn, fp, fn, tp = cm_hold.ravel()
fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

plot_confusion_matrix(
    cm_hold,
    f"Holdout Confusion Matrix - {best_name} (threshold {best_threshold:.3f})",
)

print(f"True negatives  (TN) = {tn}  correctly not flagged, did not churn")
print(f"False positives (FP) = {fp}  flagged, but did not churn")
print(f"False negatives (FN) = {fn}  not flagged, but churned")
print(f"True positives  (TP) = {tp}  flagged, and churned")
print()
print(f"False positive rate (FPR) = {fpr:.4f}  ({fpr:.2%} of retained customers flagged)")
print(f"False negative rate (FNR) = {fnr:.4f}  ({fnr:.2%} of churners missed)")

True negatives  (TN) = 585  correctly not flagged, did not churn
False positives (FP) = 188  flagged, but did not churn
False negatives (FN) = 61  not flagged, but churned
True positives  (TP) = 218  flagged, and churned

False positive rate (FPR) = 0.2432  (24.32% of retained customers flagged)
False negative rate (FNR) = 0.2186  (21.86% of churners missed)


### Result (computed from data)

Holdout confusion matrix: **TN=585, FP=188, FN=61, TP=218**. In business
terms, of 279 holdout churners the model flagged 218 (78.1% recall) and missed
61; of 773 retained customers it wrongly flagged 188 (24.3% FPR). The retained
budget concern (FPs) and missed-churner concern (FNs) are both visible, and
shifting the threshold trades one against the other — the chosen 0.525 balances
them by F1.

## Leakage Audit

The following safeguards were applied to prevent information leakage from the
holdout set into any modeling decision:

- **Holdout split before model selection.** The 15% holdout was separated in
  section 5, before any model fitting, and was never passed to a
  model-selection function.
- **Stratification during splitting.** Both the train/temporary and
  dev/holdout splits were stratified on `Churn`, preserving class proportions
  (train 26.49%, dev 26.45%, holdout 26.52%).
- **Duplicate handling before split.** Exact duplicates were removed from a
  modeling copy *before* splitting, so no identical record appears in more
  than one partition.
- **Preprocessing fitted only on training data.** The `ColumnTransformer`
  (scalers, imputers, one-hot encoders) lives inside each model pipeline and
  was fit by `GridSearchCV` on `X_train` only.
- **Hyperparameters selected using train/dev only.** Grid search used 5-fold
  stratified CV on the training set; model choice used dev F1.
- **Threshold selected using dev only.** Every threshold was chosen to
  maximize dev F1; the holdout was never consulted.
- **Holdout evaluated only after final selection.** The holdout was scored
  exactly once, in section 17, after model + hyperparameters + threshold were
  finalized.
- **No holdout-derived decisions fed back.** No holdout metric influenced any
  earlier decision in this notebook.

These rules were implemented as coded above; if any step had violated them,
the notebook would need to be fixed before results are trusted.

## 22. Final Results Table

Development-set model comparison and final holdout performance are kept
strictly separate: the holdout was used for the selected model only.

In [18]:
dev_table = dev_comparison.reset_index().rename(columns={"index": "Model"})
print("Development-set model comparison (threshold chosen per model on dev):")
display(dev_table.round(4))

final_table = pd.DataFrame([{
    "Model": best_name,
    "ROC-AUC": holdout_report["roc_auc"],
    "PR-AUC": holdout_report["pr_auc"],
    "Precision": holdout_report["precision"],
    "Recall": holdout_report["recall"],
    "F1": holdout_report["f1"],
    "Accuracy": holdout_report["accuracy"],
    "Threshold": best_threshold,
}])
print("Final holdout performance (selected model, threshold fixed from dev):")
display(final_table.round(4))

Development-set model comparison (threshold chosen per model on dev):


,Model,ROC-AUC,PR-AUC,Precision,Recall,F1,Accuracy,Threshold
0,Logistic Regression,0.8454,0.6376,0.5635,0.7338,0.6375,0.7793,0.575
1,Random Forest,0.8436,0.6406,0.5541,0.7554,0.6393,0.7745,0.450
2,XGBoost,0.8512,0.6605,0.5518,0.7662,0.6416,0.7735,0.525


Final holdout performance (selected model, threshold fixed from dev):


,Model,ROC-AUC,PR-AUC,Precision,Recall,F1,Accuracy,Threshold
0,XGBoost,0.853,0.6571,0.5369,0.7814,0.6365,0.7633,0.525


## Model Artifact — Persisted Pipeline

The final selected model — the complete fitted **preprocessing + XGBoost
pipeline** — is persisted as a versioned artifact so that downstream notebooks
(e.g. Notebook 04) load the exact model instead of retraining it.

**What is persisted.** The full fitted `Pipeline` (the `ColumnTransformer`
preprocessing plus the selected `XGBClassifier`) — i.e. the exact estimator
used for the final holdout evaluation above. The preprocessing pipeline is
persisted *with* the estimator because raw customer features must be
transformed identically everywhere downstream; persisting only the raw XGBoost
estimator would allow preprocessing drift.

**Where it is stored.** `models/telco_churn_xgboost.joblib`, plus
`models/telco_churn_xgboost_metadata.json` (model, hyperparameter, split, and
holdout metadata) and `models/telco_churn_xgboost.joblib.sha256` (a SHA-256
checksum for detecting accidental artifact replacement).

**How Notebook 04 consumes it.** Notebook 04 loads the artifact, extracts the
fitted preprocessor and XGBoost estimator through the standard pipeline
structure, verifies that the loaded model reproduces the Notebook 03 holdout
metrics, and computes SHAP values from that exact estimator.

**Why this prevents preprocessing drift.** The fitted `ColumnTransformer`
(scalers, imputers, one-hot encoders) is serialized inside the artifact, so any
consumer is guaranteed to transform features exactly as training time did.
Retraining or rebuilding the preprocessing elsewhere could silently change the
feature space; loading the artifact cannot.


In [19]:
import hashlib
import json
import joblib
import sklearn
import xgboost

MODEL_DIR = DATA_PATH.parents[2] / "models"
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "telco_churn_xgboost.joblib"
METADATA_PATH = MODEL_DIR / "telco_churn_xgboost_metadata.json"
HASH_PATH = MODEL_DIR / "telco_churn_xgboost.joblib.sha256"

final_pipeline = final_search.best_estimator_
print("Persisting final pipeline:", type(final_pipeline).__name__)
print("Pipeline steps:", list(final_pipeline.named_steps.keys()))

joblib.dump(final_pipeline, MODEL_PATH)


def compute_file_hash(path, algorithm="sha256"):
    """Return the hex digest of a file, streaming to bound memory use."""
    digest = hashlib.new(algorithm)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


artifact_hash = compute_file_hash(MODEL_PATH)
with open(HASH_PATH, "w") as f:
    f.write(artifact_hash + "\n")

transformed_feature_names = final_pipeline.named_steps["preprocess"].get_feature_names_out()

metadata = {
    "artifact": "models/telco_churn_xgboost.joblib",
    "model_type": "Pipeline(ColumnTransformer preprocessing + XGBClassifier)",
    "model_name": best_name,
    "random_state": RANDOM_STATE,
    "selected_hyperparameters": {
        k.split("__", 1)[1]: v
        for k, v in xgb_search.best_params_.items()
        if k.startswith("classifier__")
    },
    "scale_pos_weight": float(scale_pos_weight),
    "decision_threshold": round(float(best_threshold), 3),
    "dataset_path": "data/cleaned/telco_churn_clean.csv",
    "duplicate_handling": (
        "22 exact duplicate rows removed from a modeling copy before splitting; "
        "source file unchanged"
    ),
    "training_row_count": int(len(X_train)),
    "dev_row_count": int(len(X_dev)),
    "holdout_row_count": int(len(X_holdout)),
    "feature_count_before_preprocessing": int(X.shape[1]),
    "transformed_feature_count": int(len(transformed_feature_names)),
    "package_versions": {
        "xgboost": xgboost.__version__,
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "sha256": artifact_hash,
    "holdout_evaluation": {
        "roc_auc": float(holdout_report["roc_auc"]),
        "pr_auc": float(holdout_report["pr_auc"]),
        "precision": float(holdout_report["precision"]),
        "recall": float(holdout_report["recall"]),
        "f1": float(holdout_report["f1"]),
        "accuracy": float(holdout_report["accuracy"]),
        "threshold": round(float(best_threshold), 3),
        "confusion_matrix": [int(x) for x in holdout_report["confusion_matrix"].ravel()],
    },
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("Artifact written:", MODEL_PATH)
print("Metadata written:", METADATA_PATH)
print("SHA-256:", artifact_hash)
print("Artifact size (bytes):", MODEL_PATH.stat().st_size)

# Integrity check: reload the artifact and confirm identical holdout predictions.
reloaded_pipeline = joblib.load(MODEL_PATH)
proba_reloaded = reloaded_pipeline.predict_proba(X_holdout)[:, 1]
max_diff = float(np.abs(proba_reloaded - proba_holdout).max())
print("Reloaded pipeline max |holdout probability diff|:", f"{max_diff:.2e}")
assert max_diff < 1e-8
print("Artifact reload integrity check passed.")


Persisting final pipeline: Pipeline
Pipeline steps: ['preprocess', 'classifier']
Artifact written: /Users/vaibhavvikasranjan/Downloads/telco-churn-analysis/models/telco_churn_xgboost.joblib
Metadata written: /Users/vaibhavvikasranjan/Downloads/telco-churn-analysis/models/telco_churn_xgboost_metadata.json
SHA-256: 79cfdceeda66e597ed406f478c1d25cd28da242294bc58e87da3706bbfb9ae8f
Artifact size (bytes): 249495
Reloaded pipeline max |holdout probability diff|: 0.00e+00
Artifact reload integrity check passed.


## 23. Limitations

Honest limits of this analysis:

1. **Dataset size.** The model was built and evaluated on 7,032 customers
   (7,010 after removing exact duplicates). Small-sample estimates carry
   statistical uncertainty.
2. **Dataset representativeness.** Results may not generalize to another
   telecom company, geography, customer population, pricing structure, or time
   period.
3. **Observational data.** Predictive relationships do not establish causality
   (consistent with Notebook 02).
4. **Duplicate rows.** The cleaned dataset contains 22 exact duplicate rows.
   They were not silently removed from the source file; for modeling they were
   dropped from a working copy with a documented rationale. This is a modeling
   choice, not a data-fixing operation.
5. **Threshold / business costs.** The decision threshold maximizes F1, which
   weights false positives and false negatives equally. A real deployment that
   knows the financial cost of a missed churner vs a wasted intervention
   should re-select the threshold using those costs; F1 is a reasonable
   default, not a cost-optimal rule.
6. **Holdout size.** The 15% holdout (1,052 customers) provides an independent
   final estimate but is still a single realization; confidence intervals and
   repeated evaluation would add robustness.
7. **Hyperparameter search scope.** Grids were intentionally small to keep the
   search computationally reasonable; larger searches could find slightly
   better configurations.

No claim is made beyond what the executed code produced.